[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/02_load_and_filter.ipynb)

# R2-Q2 Week 2 — Load R2-Q1's outputs and filter to your working set

This notebook turns your Week-1 commitments into a concrete working data set. It loads four R2-Q1 outputs — the PlantDoc prediction table (`pd_predictions.parquet`), R2-Q1's class-space mappings (from `precommit.json`), the PV-internal predictions table for reference, and the path to the trained ResNet-18 classifier (N03 loads the weights) — then filters PD predictions to the cases where the model got the disease category wrong and runs exploratory analysis on the resulting set. The Week 2 deliverable is `working_set.parquet`: the annotated misclassification table that N03 (Grad-CAM and sanity checks) and N04 (taxonomy application) both consume.

By the end of this notebook you will have:

- A `working_set.parquet` file in your output directory — about 80 rows, each a PlantDoc image the PlantVillage-trained model misclassified at the disease-category level. Each row carries the PD class label, the PV-class prediction, both class spaces' category labels, the model's confidence, the PD host and disease metadata, and the PV-class integer index N03's Grad-CAM step needs.
- An exploratory picture of the misclassification set: category-level distribution (the basis for N04's eventual taxonomy distribution), per-host and per-class breakdowns (which your aggregation-level pre-commit in N01 already marked as hypothesis-only), and the model-confidence distribution among the wrong predictions.
- Figures from those analyses, sized and styled to drop into the Week-4 presentation.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive and set up irilab2026.
import irilab2026 as iri
iri.setup(gpu_required=False)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 1) Load the N01 pre-commit

`precommit.json` is the four-field commitment file you wrote in Notebook 01 — your taxonomy, your categorization procedure, your aggregation level, and your sanity-check criterion. The rest of this notebook doesn't read individual fields from it (those get used in Notebooks 03 and 04); what matters here is that the file exists and is well-formed. If it doesn't exist, the most likely reason is that Notebook 01 wasn't run, and the error below says so.

In [ ]:
### 1.1) Defensive load of precommit.json

import json

try:
    with open(OUTPUT_DIR / "precommit.json") as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        "Could not find precommit.json in OUTPUT_DIR. "
        "This file is produced by 01_orientation_and_precommit.ipynb — "
        "run that notebook first."
    ) from None

print(f"precommit.json loaded from {OUTPUT_DIR / 'precommit.json'}")
print(f"  top-level keys: {sorted(precommit.keys())}")

## 2) Read R2-Q1's outputs

R2-Q2 inherits four artifacts from R2-Q1, all sitting in
`iri.output_dir("r2_q1")`:

| File                       | What it is                                  | This notebook's use                         |
|----------------------------|---------------------------------------------|---------------------------------------------|
| `precommit.json`           | R2-Q1's Week 1 pre-commit                   | Source of the two class-space mappings      |
| `pd_predictions.parquet`   | 236-row PlantDoc prediction table           | Becomes the working set after filtering     |
| `pv_predictions.parquet`   | PlantVillage-internal prediction table      | Available for reference; not central        |
| `baseline_resnet18.pt`     | The trained PlantVillage ResNet-18          | Path captured here; weights loaded in N03   |

Two code cells follow. The first reads R2-Q1's precommit and pulls out
the category vocabulary and the two class-space mappings. It also
verifies that `baseline_resnet18.pt` is on disk — N03 will need to load
its weights, and confirming the file exists now (rather than four cells
into N03) is the gentler failure mode. The second reads the two
prediction tables: `pd_predictions.parquet` is required (this notebook
fails if it's missing), `pv_predictions.parquet` is optional (a warning
prints if it's missing and the notebook proceeds).

In [ ]:
import json

R2Q1_DIR = iri.output_dir("r2_q1")

# Load R2-Q1's precommit.
try:
    with open(R2Q1_DIR / "precommit.json") as f:
        precommit_r2q1 = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find precommit.json in {R2Q1_DIR}. "
        "This file is produced by R2-Q1's 02_pd_orientation.ipynb — "
        "run R2-Q1 to completion before running this notebook."
    ) from None

# Pull out the dual-mapping block.
remapping = precommit_r2q1["class_space_remapping"]
pv_class_to_category = remapping["pv_class_to_category"]
pd_class_to_category = remapping["pd_class_to_category"]
categories = remapping["categories_used"]

# Confirm the trained classifier is on disk.
# N03 loads the weights; here we only verify the file exists so the
# failure (if any) surfaces now rather than four cells into N03.
classifier_path = R2Q1_DIR / "baseline_resnet18.pt"
if not classifier_path.exists():
    raise FileNotFoundError(
        f"Could not find baseline_resnet18.pt in {R2Q1_DIR}. "
        "This file is produced by R2-Q1's 03_pv_classifier.ipynb."
    )

print(f"R2-Q1 precommit loaded from {R2Q1_DIR / 'precommit.json'}")
print(f"  PV mapping:    {len(pv_class_to_category)} classes")
print(f"  PD mapping:    {len(pd_class_to_category)} classes")
print(f"  Categories:    {categories}")
print(f"Classifier path: {classifier_path}")
print(f"  size:          {classifier_path.stat().st_size / 1e6:.1f} MB")

In [ ]:
import pandas as pd

# pd_predictions: required.
try:
    pd_predictions = pd.read_parquet(R2Q1_DIR / "pd_predictions.parquet")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find pd_predictions.parquet in {R2Q1_DIR}. "
        "This file is produced by R2-Q1's 04_transfer_gap_measurement.ipynb."
    ) from None

print(f"pd_predictions: {len(pd_predictions):,} rows, "
      f"{len(pd_predictions.columns)} cols")
print(f"  columns: {list(pd_predictions.columns)}")

# pv_predictions: optional. Available for cross-checks if present;
# this notebook does not depend on it.
try:
    pv_predictions = pd.read_parquet(R2Q1_DIR / "pv_predictions.parquet")
    print(f"pv_predictions: {len(pv_predictions):,} rows, "
          f"{len(pv_predictions.columns)} cols")
except FileNotFoundError:
    pv_predictions = None
    print("pv_predictions: not found (optional; proceeding without it)")

## 4) Filter to misclassifications

R2-Q2's working set is the subset of PlantDoc test images that the
PlantVillage-trained model got *wrong at category level* — where the
predicted disease category does not match the true disease category.
R2-Q1's N04 already computed this judgment per row in
`correct_at_category` (a boolean column in `pd_predictions.parquet`).
This section keeps the rows where that column is `False`.

Two columns are renamed in the process. R2-Q1 used `true_label` and
`pred_label` because there was one inference pass and the class space
was unambiguous within that notebook. R2-Q2 lives across two class
spaces — PlantDoc on the ground-truth side, PlantVillage on the
prediction side — and every downstream notebook reads from both. The
rename to `true_pd_class` and `pred_pv_class` makes the class space
visible at every read site, which matters when N03 needs the PV class
index for Grad-CAM and N04 needs the PD class for per-host cuts.

In [ ]:
# Filter to category-level misclassifications.
misclassified = pd_predictions[~pd_predictions["correct_at_category"]].copy()

# Rename for class-space clarity.
misclassified = misclassified.rename(columns={
    "true_label": "true_pd_class",
    "pred_label": "pred_pv_class",
})

# Drop the now-redundant correct_at_category column — every row is False
# by construction.
misclassified = misclassified.drop(columns=["correct_at_category"])

print(f"Misclassifications: {len(misclassified):,} of "
      f"{len(pd_predictions):,} PD test images "
      f"({len(misclassified) / len(pd_predictions):.1%})")
print(f"Columns: {list(misclassified.columns)}")

## 5) Category-level distribution

The misclassification set has two category-level axes worth looking at
separately:

- **True category.** Which categories' images the model fails on. A
  category with many misclassifications could mean either that the
  model handles it badly, or that PlantDoc happens to have many images
  in that category.
- **Predicted category.** Which categories the model assigns its wrong
  answers to. Concentration here suggests the model is biased toward
  some categories regardless of what's actually in front of it.

Two figures follow. The first is a side-by-side bar chart of counts
along each axis. The second is a confusion heatmap — true category on
one axis, predicted on the other, raw counts in each cell. Counts only,
not rates: with the off-diagonal cells in the single-digit range,
row-normalized rates would turn small counts into misleading
percentages.

Per your N01 pre-commit, the only category-level result that goes into
the paper's *results* section is the overall failure-mode taxonomy
distribution (which N04 produces). The category counts shown here
inform the *discussion* — they're context for the taxonomy distribution,
not findings in their own right.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")

In [ ]:
# Pull the colorblind palette as a list (one color per category).
palette_list = sns.color_palette("colorblind", n_colors=len(categories))

# categories is alphabetical: ['bacterial', 'fungal', 'healthy', 'pest', 'viral']
# Swap positions 0 (bacterial) and 1 (fungal).
palette_list[0], palette_list[1] = palette_list[1], palette_list[0]

true_counts = misclassified["true_category"].value_counts().reindex(
    categories, fill_value=0
)
pred_counts = misclassified["pred_category"].value_counts().reindex(
    categories, fill_value=0
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

sns.barplot(
    x=true_counts.index, y=true_counts.values,
    hue=true_counts.index, palette=palette_list,
    order=categories, legend=False, ax=axes[0],
)
axes[0].set_title("Misclassifications by true category")
axes[0].set_xlabel("True category")
axes[0].set_ylabel("Count")

sns.barplot(
    x=pred_counts.index, y=pred_counts.values,
    hue=pred_counts.index, palette=palette_list,
    order=categories, legend=False, ax=axes[1],
)
axes[1].set_title("Misclassifications by predicted category")
axes[1].set_xlabel("Predicted category")
axes[1].set_ylabel("")

fig.tight_layout()
plt.show()

print("By true category:")
print(true_counts.to_string())
print()
print("By predicted category:")
print(pred_counts.to_string())

In [ ]:
confusion = (
    misclassified.groupby(["true_category", "pred_category"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=categories, columns=categories, fill_value=0)
)

# Mask the diagonal — by construction, all rows are misclassifications,
# so the diagonal is zero everywhere. Showing the zeros would be noise.
mask = pd.DataFrame(
    [[i == j for j in range(len(categories))] for i in range(len(categories))],
    index=categories,
    columns=categories,
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    confusion,
    annot=True,
    fmt="d",
    mask=mask,
    cmap="Reds",
    cbar_kws={"label": "Count"},
    ax=ax,
)
ax.set_title("Misclassification confusion: true → predicted category")
ax.set_xlabel("Predicted category")
ax.set_ylabel("True category")

fig.tight_layout()
plt.show()

> **Hypothesis vs. finding.** The cuts below — per-host and per-PD-class
> counts — produce per-bucket numbers in the single digits. They are
> useful for spotting patterns worth investigating, not for reporting
> as findings. Your N01 pre-commit named the *overall taxonomy
> distribution* as the only category-level result that goes into the
> paper's results section; per-host and per-class cuts inform the
> discussion, with the small-sample fragility flagged. Treat what
> follows as hypothesis-generating.

## 6) Per-host and per-PD-class breakdowns

Two more axes worth looking at on the misclassification set:

- **Host.** PlantDoc covers 13 host species. A host with many
  misclassifications could mean the model handles that host's leaf
  morphology badly, or it could mean PD happens to have many images of
  that host.
- **PD class.** PlantDoc's 28 classes are host-disease pairings
  (e.g., `Apple Scab Leaf`, `Tomato leaf bacterial spot`). Per-class
  counts surface which specific pairings drive the misclassifications.

Both charts are horizontal — the labels are long enough that horizontal
orientation reads more cleanly than rotated tick labels on a vertical
axis. Hosts above, classes below, sorted descending by count in each
case so the largest contributors are at the top.

In [ ]:
host_counts = misclassified["host"].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    x=host_counts.values, y=host_counts.index,
    hue=host_counts.index, palette="colorblind",
    orient="h", legend=False, ax=ax,
)
ax.set_title("Misclassifications by host")
ax.set_xlabel("Count")
ax.set_ylabel("Host")

fig.tight_layout()
plt.show()

print(host_counts.to_string())

In [ ]:
class_counts = misclassified["true_pd_class"].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(
    x=class_counts.values, y=class_counts.index,
    hue=class_counts.index, palette="colorblind",
    orient="h", legend=False, ax=ax,
)
ax.set_title("Misclassifications by true PD class")
ax.set_xlabel("Count")
ax.set_ylabel("True PD class")

fig.tight_layout()
plt.show()

print(class_counts.to_string())

## 7) Confidence on the misclassifications

Every row in the misclassification set carries a `confidence` score —
the softmax probability the model assigned to the (wrong) class it
predicted. The distribution of these scores answers a different
question from the category and host cuts above: *how sure was the model
when it was wrong?*

A misclassification with low confidence is roughly what you'd expect —
the model was uncertain, and it landed off the right answer. A
misclassification with high confidence is more interesting: the model
was sure, and it was sure wrong. These confidently-wrong cases are
where the failure-mode analysis in N03 (Grad-CAM) and N04 (taxonomy
categorization) is likely to be most informative — when the model
commits to a wrong answer, *what visual evidence was it committing to?*

A histogram of the confidence distribution follows, then a table of
the ten most confidently-wrong predictions. The table's `filename`
column is the join key N03 will use to pull specific images for
Grad-CAM — long strings, ignore them visually, but they're how a
specific row in this table becomes a specific heatmap in N03.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(misclassified["confidence"], bins=20, ax=ax)
ax.set_title("Confidence distribution on misclassifications")
ax.set_xlabel("Confidence (softmax probability of predicted class)")
ax.set_ylabel("Count")
ax.set_xlim(0, 1)

fig.tight_layout()
plt.show()

print(f"Confidence summary on {len(misclassified)} misclassifications:")
print(f"  median:  {misclassified['confidence'].median():.3f}")
print(f"  mean:    {misclassified['confidence'].mean():.3f}")
print(f"  min:     {misclassified['confidence'].min():.3f}")
print(f"  max:     {misclassified['confidence'].max():.3f}")
print(f"  >= 0.9:  {(misclassified['confidence'] >= 0.9).sum()} rows")
print(f"  >= 0.5:  {(misclassified['confidence'] >= 0.5).sum()} rows")